# 2.A: Profiles of bike trips
Goal: find groups of trips that are ridden in the same way, then explain what each group means.

Data: `trips_small_cleaned.csv`, `cities_clean.csv`, `cleaned_stations.csv`, `bikes.csv`, `cleaned_bike_types.csv`.

In [ ]:
import folium
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from folium.plugins import HeatMap
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, davies_bouldin_score, silhouette_score
from sklearn.neighbors import NearestCentroid
from sklearn.preprocessing import StandardScaler

COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#6250d6"]   # one colour per cluster
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": "#e6e5e0"})

## 1. Load the data and build the trip information

In [ ]:
trips = pd.read_csv("../trips_small_cleaned.csv")
cities = pd.read_csv("../cities_clean.csv").rename(columns={"id": "city_id", "name": "city"})
bikes = pd.read_csv("../bikes.csv")
bike_types = pd.read_csv("../cleaned_bike_types.csv")
stations = pd.read_csv("../cleaned_stations.csv")


def km_between(lat1, lon1, lat2, lon2):
    """Distance in km between two points (latitude / longitude in degrees)."""
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * 6371 * np.arcsin(np.sqrt(a))


# time: time_start is in UTC seconds -> local time of the city
trips = trips.merge(cities[["city_id", "city", "country", "timezone"]], on="city_id")
utc = pd.to_datetime(trips["time_start"], unit="s", utc=True)
trips["local_time"] = pd.concat([utc[g.index].dt.tz_convert(tz).dt.tz_localize(None)
                                 for tz, g in trips.groupby("timezone")])
trips["hour"] = trips["local_time"].dt.hour + trips["local_time"].dt.minute / 60
trips["weekend"] = trips["local_time"].dt.dayofweek >= 5
trips["summer"] = trips["local_time"].dt.month.isin([5, 6, 7, 8])

# trip shape: distance is a straight line, so speed = how direct the trip is
trips["speed_kmh"] = trips["distance"] / trips["duration"] * 3.6

# position in the city: the centre is the median start point of the city's trips
centre = trips.groupby("city_id")[["lat_start", "lon_start"]].transform("median")
trips["start_to_centre_km"] = km_between(trips["lat_start"], trips["lon_start"], centre["lat_start"], centre["lon_start"])
trips["end_to_centre_km"] = km_between(trips["lat_end"], trips["lon_end"], centre["lat_start"], centre["lon_start"])
trips["away_from_centre_km"] = trips["end_to_centre_km"] - trips["start_to_centre_km"]   # < 0: goes towards the centre

# bike type: e-bike, cargo bike
bike_info = bikes.merge(bike_types, left_on="bike_type_id", right_on="id", suffixes=("", "_type")).set_index("id")
trips["electric"] = trips["bike_id"].map(bike_info["is_electric"]).fillna(False).astype(bool)
trips["cargo"] = trips["bike_id"].map(bike_info["is_cargo"]).fillna(False).astype(bool)

# station size = number of racks (station not in the table -> unknown, not 0; 0 racks may be a virtual station)
racks = stations.set_index("id")[["bike_racks", "special_racks"]].fillna(0).sum(axis=1)
trips["start_racks"] = trips["station_id_start"].map(racks)
trips["start_no_racks"] = trips["start_racks"] == 0

# bike history: previous trip of the same bike
trips = trips.sort_values(["bike_id", "time_start"]).reset_index(drop=True)
before = trips.groupby("bike_id").shift()
trips["parked_hours"] = (trips["time_start"] - before["time_start"] - before["duration"]) / 3600
trips["moved_by_operator"] = km_between(before["lat_end"], before["lon_end"], trips["lat_start"], trips["lon_start"]) > 0.2

print(f"{len(trips):,} trips in {trips['city_id'].nunique()} cities and {trips['country'].nunique()} countries, "
      f"{trips['local_time'].min():%Y-%m-%d} to {trips['local_time'].max():%Y-%m-%d}")
print(f"start station known: {trips['start_racks'].notna().mean():.1%} | start at a place with 0 racks: "
      f"{trips['start_no_racks'].mean():.1%} | cargo bikes: {trips['cargo'].mean():.2%} of trips")

## 2. Features used for clustering
We only use how the trip is ridden. Time of year, city and bike type are kept for later, to explain the groups.

| feature | meaning |
|---|---|
| `duration_min` | how long the trip takes (minutes) |
| `distance_km` | straight-line distance from start to end (km) |
| `speed_kmh` | distance / duration: low = slow trip with stops, high = direct trip |
| `hour_sin`, `hour_cos` | time of day on a circle (23 h is next to 0 h) |
| `start_to_centre` | distance from the start to the city centre, divided by the city size (1 = edge of the city) |
| `away_from_centre_km` | change of distance to the centre (km): > 0 the trip goes away from the centre, < 0 towards it |

City size = distance to the centre that 90 % of the trips start within (at least 0.5 km).
The 1 % most extreme values of duration, distance and `away_from_centre_km` are cut, so a few very long trips do not
pull the groups. Then every feature is put on the same scale (mean 0, standard deviation 1).

In [ ]:
def cut_extremes(s):
    return s.clip(*s.quantile([0.01, 0.99]))


city_size = trips.groupby("city_id")["start_to_centre_km"].transform(lambda s: s.quantile(0.9)).clip(lower=0.5)
features = pd.DataFrame({
    "duration_min": cut_extremes(trips["duration"] / 60),
    "distance_km": cut_extremes(trips["distance"] / 1000),
    "speed_kmh": trips["speed_kmh"],
    "hour_sin": np.sin(2 * np.pi * trips["hour"] / 24),
    "hour_cos": np.cos(2 * np.pi * trips["hour"] / 24),
    "start_to_centre": (trips["start_to_centre_km"] / city_size).clip(0, 3),
    "away_from_centre_km": cut_extremes(trips["away_from_centre_km"]),
})
X = StandardScaler().fit_transform(features)
X[:, 3:5] /= np.sqrt(2)   # hour uses 2 columns: give each one less weight so hour counts as ONE feature

rng = np.random.default_rng(42)
score_idx = rng.choice(len(X), 10_000, replace=False)   # same 10 000 trips to score every model
X_score = X[score_idx]
features.describe().T.round(2)

### Check the scaling: variance of each feature, and PCA eigenvalues
Before scaling, the features have very different variances (duration in minutes varies much more than `hour_sin`),
so they would not count the same in the distances between trips. After scaling, every feature has variance 1
(`hour_sin` and `hour_cos` 0.5 each, so the hour counts as one feature).
The PCA eigenvalues show how much of the total variance each new direction carries.

In [ ]:
eigen = PCA().fit(X)
fig, ax = plt.subplots(1, 3, figsize=(16, 3.8))
features.var().plot.bar(ax=ax[0], color=COLORS[0], logy=True, title="variance before scaling (log scale)")
pd.Series(X.var(axis=0), index=features.columns).plot.bar(ax=ax[1], color=COLORS[0], title="variance after scaling")
ax[2].bar(range(1, X.shape[1] + 1), eigen.explained_variance_, color=COLORS[0])
for i, (value, total) in enumerate(zip(eigen.explained_variance_, eigen.explained_variance_ratio_.cumsum())):
    ax[2].text(i + 1, value, f"{total:.0%}", ha="center", va="bottom", fontsize=9)
ax[2].set(title="PCA eigenvalues (label = cumulative % of variance)", xlabel="component")
plt.tight_layout()
plt.show()
pd.DataFrame({"variance before": features.var(), "variance after": X.var(axis=0)}).round(3)

### Check the correlation between the features
K-Means and Ward use distances between trips: if two features carry the same information, it counts twice.
We look at the correlation between features and at the VIF (variance inflation factor: how well a feature can be
predicted from the others; above 5 means strong redundancy).

In [ ]:
corr = features.corr()
plt.figure(figsize=(7, 5.5))
plt.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
plt.xticks(range(len(corr)), corr.columns, rotation=45, ha="right")
plt.yticks(range(len(corr)), corr.columns)
for (i, j), v in np.ndenumerate(corr.to_numpy()):
    plt.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8, color="white" if abs(v) > 0.6 else "black")
plt.colorbar(label="correlation")
plt.grid(False)
plt.title("Correlation between the features")
plt.show()
pd.Series(np.diag(np.linalg.inv(corr)), index=corr.columns, name="VIF").round(2)

The strongest links are duration – speed (−0.49), start position – direction (−0.49: trips that start far out can
mostly only go inwards) and duration – distance (0.39). All VIF values are below 2.5, far from 5: the features are
linked but each one still brings its own information, so we keep all of them.

## 3. The two models
- **K-Means**: puts `k` centres in the data, gives each trip to the nearest centre, moves each centre to the middle of
  its trips, and repeats. Fast, so it is fitted on 200 000 trips; then every trip goes to its nearest centre.
- **Ward**: starts with each trip alone and joins, step by step, the two groups that stay the most compact.
  It needs the distance between every pair of trips, so it is fitted on 20 000 trips (about 3 GB of memory).
  Ward has no rule for new trips: every trip outside the sample goes to the nearest Ward group centre.

In [ ]:
def fit_kmeans(idx, k, seed=0):
    return KMeans(n_clusters=k, n_init=10, random_state=seed).fit(X[idx])


def fit_ward(idx, k, seed=0):
    labels = fcluster(linkage(X[idx], method="ward"), t=k, criterion="maxclust") - 1
    return NearestCentroid().fit(X[idx], labels)   # new trips go to the nearest group centre


def sample(size, seed):
    return np.random.default_rng(seed).choice(len(X), size, replace=False)

### Choosing the number of groups `k` for K-Means
For each `k`, K-Means is fitted on 5 different random samples of 100 000 trips. We look at:
- **inertia** (elbow): how tight the groups are, look for the bend of the curve;
- **silhouette** (higher = groups better separated) and **Davies–Bouldin** (lower = better);
- **stability**: do the 5 samples give the same groups? (adjusted Rand index, 1 = identical).

Rule: among the stable `k` (stability ≥ 0.95), keep the one with the lowest Davies–Bouldin.

In [ ]:
rows = []
for k in range(2, 9):
    runs = [fit_kmeans(sample(100_000, 100 + s), k, s) for s in range(5)]
    groups = [m.predict(X_score) for m in runs]
    rows.append({"k": k, "inertia": runs[0].inertia_,
                 "silhouette": np.mean([silhouette_score(X_score, g) for g in groups]),
                 "davies_bouldin": np.mean([davies_bouldin_score(X_score, g) for g in groups]),
                 "stability": np.mean([adjusted_rand_score(a, b) for i, a in enumerate(groups) for b in groups[i + 1:]])})
k_scores = pd.DataFrame(rows).set_index("k")
K = int(k_scores.loc[k_scores["stability"] >= 0.95, "davies_bouldin"].idxmin())

fig, ax = plt.subplots(1, 4, figsize=(17, 3.5))
for a, col in zip(ax, k_scores.columns):
    a.plot(k_scores.index, k_scores[col], marker="o", color=COLORS[0])
    a.axvline(K, color="grey", ls="--")
    a.set(title=col, xlabel="k")
plt.tight_layout()
plt.show()
print("chosen k =", K)
k_scores.round(3)

### Fit the two models
K-Means groups are numbered by size (group 0 = the biggest). Ward groups get the number of the K-Means group they
share the most trips with, so the colours match.

In [ ]:
kmeans = fit_kmeans(sample(200_000, 1), K)
order = np.argsort(np.argsort(-np.bincount(kmeans.predict(X))))     # new number of each group, by size
trips["kmeans"] = order[kmeans.predict(X)]

ward_idx = sample(20_000, 3)
tree = linkage(X[ward_idx], method="ward")
trips["ward"] = NearestCentroid().fit(X[ward_idx], fcluster(tree, t=K, criterion="maxclust") - 1).predict(X)
rows_, cols_ = linear_sum_assignment(-pd.crosstab(trips["ward"], trips["kmeans"]).to_numpy())
trips["ward"] = trips["ward"].map(dict(zip(rows_, cols_)))

plt.figure(figsize=(12, 3.5))
dendrogram(tree, truncate_mode="lastp", p=30, no_labels=True, color_threshold=tree[-K + 1, 2])
plt.title(f"Ward tree (last 30 joins), cut into {K} groups")
plt.show()

for model in ["kmeans", "ward"]:
    print(model, trips[model].value_counts(normalize=True).sort_index().round(2).to_dict())

### Compare the models
- **silhouette** and **Davies–Bouldin** on the same 10 000 trips;
- **agreement**: adjusted Rand index between two models (1 = same groups, 0 = no link);
- **stability**: each model is fitted again on other random samples, to check whether the groups stay the same.

In [ ]:
models = {"K-Means": ("kmeans", fit_kmeans, 200_000, 5), "Ward": ("ward", fit_ward, 20_000, 5)}
labels = {name: trips[col].to_numpy()[score_idx] for name, (col, *_) in models.items()}

summary = []
for name, (col, fit, size, n_runs) in models.items():
    groups = [fit(sample(size, 200 + s), K, s).predict(X_score) for s in range(n_runs)]
    summary.append({"model": name, "silhouette": silhouette_score(X_score, labels[name]),
                    "davies_bouldin": davies_bouldin_score(X_score, labels[name]),
                    "stability": np.mean([adjusted_rand_score(a, b) for i, a in enumerate(groups) for b in groups[i + 1:]])})
print("agreement between K-Means and Ward:", round(adjusted_rand_score(labels["K-Means"], labels["Ward"]), 2))
pd.DataFrame(summary).set_index("model").round(3)

### 2D picture of the trips (PCA)
PCA finds the 2 directions where the trips differ the most. Each axis is a mix of features; the label shows the
3 biggest weights (for example `+0.7·duration_min` = longer trips are further right).

In [ ]:
pca = PCA(n_components=2).fit(X_score)
points = pca.transform(X_score)
weights = pd.DataFrame(pca.components_.T, index=features.columns, columns=["PC1", "PC2"])


def axis_name(i):
    top = weights.iloc[:, i].abs().sort_values(ascending=False).index[:3]
    mix = "  ".join(f"{weights.iloc[:, i][f]:+.2f}·{f}" for f in top)
    return f"PC{i + 1} ({pca.explained_variance_ratio_[i]:.0%} of the variation)\n≈ {mix}"


fig, ax = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
for a, (name, lab) in zip(ax, labels.items()):
    for g in np.unique(lab):
        a.scatter(*points[lab == g].T, s=4, alpha=0.5, color=COLORS[g], label=f"group {g}")
    a.set_title(name)
    a.legend(markerscale=3, fontsize=8)
fig.supxlabel(axis_name(0), fontsize=10)
fig.supylabel(axis_name(1), fontsize=10)
plt.tight_layout()
plt.show()
weights.round(2)

**Comparison of the models**
- **K-Means** gives the same groups on every sample (stability ≈ 0.99), with medium separation (silhouette ≈ 0.23).
- **Ward** has slightly better separation scores (silhouette ≈ 0.27, Davies–Bouldin ≈ 1.31) and partly agrees with
  K-Means (agreement ≈ 0.55). But its groups still change between samples (stability ≈ 0.63) and one group holds about
  two thirds of the trips, so it describes the trips in less detail.
- Both models only reach medium separation (silhouette 0.23–0.27). This suggests that the trips do not form
  well-separated clouds: the groups are more like zones of one continuous cloud.
- The 2D picture agrees: the first direction is mostly *where the trip goes* (away from / towards the centre, distance),
  the second mostly *how long and how slow* it is.

We use K-Means for the interpretation because its groups are the most reproducible and more balanced. The choice of k is not certain:
k = 2 is also stable and has a better silhouette, so 4 groups is a useful level of detail rather than a definitive number.

## 4. Description of the K-Means groups
Typical trip of each group (median), with information that was **not** used for clustering.
Rush hours = weekdays 7:00–9:30 and 16:00–19:00. Summer = May to August. The last column is all trips together.

In [ ]:
weekday = ~trips["weekend"]
trips["morning_rush"] = weekday & trips["hour"].between(7, 9.5)
trips["evening_rush"] = weekday & trips["hour"].between(16, 19)
describe = dict(
    share=("kmeans", "size"),
    duration_min=("duration", lambda s: s.median() / 60),
    distance_km=("distance", lambda s: s.median() / 1000),
    speed_kmh=("speed_kmh", "median"),
    start_to_centre_km=("start_to_centre_km", "median"),
    away_from_centre_km=("away_from_centre_km", "median"),
    weekend=("weekend", "mean"),
    morning_rush=("morning_rush", "mean"),
    evening_rush=("evening_rush", "mean"),
    summer=("summer", "mean"),
    electric=("electric", "mean"),
    parked_hours=("parked_hours", "median"),
    moved_by_operator=("moved_by_operator", "mean"),
    start_no_racks=("start_no_racks", "mean"),
)
profile = pd.concat([trips.groupby("kmeans").agg(**describe), trips.assign(kmeans="all trips").groupby("kmeans").agg(**describe)])
profile["share"] /= len(trips)
profile.round(2).T

In [ ]:
fig, ax = plt.subplots(1, K, figsize=(17, 3.3), sharey=True)
for g, a in enumerate(ax):
    for start, end in [(7, 9), (12, 14), (16, 19)]:    # morning, lunch, end of the school / work day
        a.axvspan(start, end, color="#eeeeee", zorder=0)
    for weekend, color in [(False, COLORS[0]), (True, COLORS[1])]:
        hours = trips.loc[(trips["kmeans"] == g) & (trips["weekend"] == weekend), "local_time"].dt.hour
        a.plot(hours.value_counts(normalize=True).sort_index(), color=color, label="weekend" if weekend else "weekday")
    a.set(title=f"group {g}", xlabel="hour", xticks=[0, 6, 12, 18])
ax[0].set_ylabel("share of the group's trips")
ax[0].legend()
plt.tight_layout()
plt.show()

**Hourly patterns** (grey bands: 7–9 h, 12–14 h, 16–19 h). The reasons for riding are not observed, so the explanations below are hypotheses.
- 7–8 h on weekdays: a clear peak for groups 0, 1 and 2; people may be going to work or to school.
- 12–14 h: no clear lunch peak; trips rise slowly during the day, so lunch-break trips, if any, do not stand out.
- 16–17 h on weekdays: the biggest peak, a bit before 18 h; it may be the end of the school day and of work.
- After 18 h: weekday trips fall, but weekend trips stay higher in the evening, possibly for evening outings.
- Weekends: no morning peak, a smooth rise up to 13–18 h, more like free time.
- Group 3 (long slow rides) starts later (about 9 h), stays high all day, and peaks at 14 h on weekends.

**Interpretation of the groups.** These are hypotheses: the data contain no rider identifier and no trip purpose.
- **Group 0 (56 %)**: short (9 min, 1.0 km) trips that start close to the centre (0.7 km) and stay there.
  Weekday peaks at about 7 h and 16 h. They look like everyday short trips in the city centre.
- **Group 1 (24 %)**: medium trips (13 min, 2.1 km) that start further out (2.1 km) and end about 1.1 km closer to the
  centre. Weekday peaks at about 7 h and, stronger, at 16 h. They look like trips towards the centre.
- **Group 2 (14 %)**: the longest direct trips (19 min, 3.1 km, 10.5 km/h). They start near the centre and end about
  2.3 km further out. More e-bikes (29 % vs 16 % for all trips) and more bikes moved by the operator before the trip
  (24 % vs 16 %). They look like trips away from the centre.
- **Group 3 (6 %)**: very long rentals (92 min) that end only 1.8 km from where they started (about 1 km/h). A bit more
  frequent at weekends (29 % vs 23 %) and in summer (41 % vs 38 %), with a weekend peak in the afternoon. They may be
  leisure rides with stops or loops, but some could also be bikes kept a long time for other reasons; the data cannot
  tell these cases apart.

### Names of the groups
The names follow from the interpretation above. The assertions check that each name still matches its group if the data change.

In [ ]:
NAMES = {0: "Short trip near the centre", 1: "Trip towards the centre", 2: "Trip away from the centre",
         3: "Long slow ride"}
groups_only = profile.drop("all trips")
assert groups_only["duration_min"].idxmax() == 3
assert groups_only["away_from_centre_km"].idxmin() == 1
assert groups_only["away_from_centre_km"].idxmax() == 2

PROFILES = [NAMES[g] for g in range(K)]
COLOR_OF = dict(zip(PROFILES, COLORS))
trips["profile"] = pd.Categorical(trips["kmeans"].map(NAMES), categories=PROFILES)
profile.rename(index=NAMES)[["share", "duration_min", "distance_km", "speed_kmh", "away_from_centre_km", "weekend", "electric"]].round(2)

## 5. Profiles by context, place and bike type
**Lift** = share of a profile in a group of trips ÷ its share in all trips. 1 = normal, 1.5 = 50 % more often than normal.

In [ ]:
groups = {"weekend": trips["weekend"], "weekday 7:00–9:30": trips["morning_rush"],
          "weekday 16:00–19:00": trips["evening_rush"], "summer": trips["summer"], "e-bike": trips["electric"],
          "bike moved by operator": trips["moved_by_operator"], "start at a place with 0 racks": trips["start_no_racks"]}
normal = trips["profile"].value_counts(normalize=True)
lift = pd.DataFrame({g: trips.loc[m, "profile"].value_counts(normalize=True) / normal for g, m in groups.items()}).T[PROFILES]

fig, a = plt.subplots(figsize=(10, 3.5))
image = a.imshow(lift.to_numpy(float), cmap="RdBu_r", vmin=0.4, vmax=1.6, aspect="auto")
a.set(xticks=range(K), xticklabels=[p.replace(" the ", "\nthe ") for p in PROFILES],
      yticks=range(len(lift)), yticklabels=lift.index)
a.grid(False)
for (i, j), v in np.ndenumerate(lift.to_numpy(float)):
    a.text(j, i, f"{v:.2f}", ha="center", va="center", color="white" if abs(v - 1) > 0.35 else "black")
plt.colorbar(image, label="lift (1 = normal)")
plt.title("More often (red) or less often (blue) than normal")
plt.tight_layout()
plt.show()

### Map of trip start points by profile
Heat map of start points on an OpenStreetMap background, centred on the city with the most trips. Each profile is a
layer that can be shown or hidden in the top-right box. The map is saved to `assets/trip_profiles_map.html`.

In [ ]:
top_city = trips["city"].value_counts().index[0]
in_city = trips[trips["city"] == top_city]
m = folium.Map(location=[in_city["lat_start"].median(), in_city["lon_start"].median()], zoom_start=11, tiles="OpenStreetMap")
for i, p in enumerate(PROFILES):
    starts = trips.loc[trips["profile"] == p, ["lat_start", "lon_start"]]
    layer = folium.FeatureGroup(name=p, show=i == 0)
    HeatMap(starts.sample(min(len(starts), 20_000), random_state=0).values.tolist(),
            radius=15, blur=10, min_opacity=0.4).add_to(layer)
    layer.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
m.save("../assets/trip_profiles_map.html")
m

### Profiles in the 15 cities with the most trips

In [ ]:
top15 = trips["city"].value_counts().index[:15]
by_city = pd.crosstab(trips["city"], trips["profile"], normalize="index").loc[top15].sort_values(PROFILES[0])
by_city.plot.barh(stacked=True, color=COLORS[:K], width=0.8, edgecolor="white", figsize=(11, 6))
plt.xlabel("share of the city's trips")
plt.legend(ncol=K, loc="lower center", bbox_to_anchor=(0.5, 1), frameon=False)
plt.show()
by_city.round(2)

## 6. Trips by country
How many trips and cities each country has, a few simple indicators, and the mix of profiles.
A country with only one or a few cities mostly shows those cities, not the whole country.

In [ ]:
by_country = trips.groupby("country").agg(
    trips=("city", "size"), cities=("city", "nunique"), duration_min=("duration", lambda s: s.median() / 60),
    weekend=("weekend", "mean"), electric=("electric", "mean"), no_racks=("start_no_racks", "mean"),
).sort_values("trips", ascending=False)
by_country["share"] = by_country["trips"] / len(trips)
mix = pd.crosstab(trips["country"], trips["profile"], normalize="index").loc[by_country.index]

fig, ax = plt.subplots(1, 2, figsize=(16, 4.5))
by_country["trips"].plot.bar(ax=ax[0], color=COLORS[0], title="Number of trips per country (label = number of cities)")
for i, (n, c) in enumerate(zip(by_country["trips"], by_country["cities"])):
    ax[0].text(i, n, c, ha="center", va="bottom", fontsize=9)
mix.plot.bar(stacked=True, ax=ax[1], color=COLORS[:K], width=0.8, edgecolor="white", title="Profile mix per country")
ax[1].legend(fontsize=8, loc="lower center", bbox_to_anchor=(0.5, 1.08), ncol=2, frameon=False)
ax[1].set(ylabel="share of the country's trips")
plt.tight_layout()
plt.show()
pd.concat([by_country.round(2), mix.round(2)], axis=1)

**Observations**

- Trips are concentrated in two countries: Poland (29 %, 43 cities) and Germany (25 %, 59 cities) hold more
  than half of the trips. Cyprus, Latvia and Malta have one city each and less than 0.3 % of the trips, so their
  numbers only describe that one city.
- The bike fleet changes a lot between countries: e-bikes are 91 % of the trips in Slovenia and 78 % in Spain,
  46 % in Croatia, but almost none in Czechia, Switzerland, Austria and Poland. This most likely reflects the bikes
  each system offers, not a choice of the riders.
- Places with 0 racks (maybe virtual stations) are almost all the start places in Switzerland (99 %), and most of
  them in Austria (75 %) and Czechia (73 %), but almost none in the UK, Spain or Slovenia: again a difference between
  systems.
- The profile mix also differs: in Czechia 73 % of the trips are short trips near the centre (median 8 min),
  while in the UK only 31 % are, with many trips towards (34 %) or away from (29 %) the centre and longer trips
  (median 18 min). The UK data come from only 4 cities, one of them a university network, which may explain part of
  this. Spain, Croatia and Slovenia have more trips away from the centre (21–22 %) and also many e-bikes; with this
  data we cannot say whether the country or the e-bikes explain it.
- Poland has the most weekend trips (28 % vs 19 % in Germany) and the most long slow rides among the big countries
  (8 %).

These differences mix several effects (city size, bike fleet, type of stations, number of cities per country), so
they describe the data we have rather than general habits of each country.

## 7. Conclusions

| profile | share | typical trip |
|---|---|---|
| Short trip near the centre | 56 % | 9 min, 1.0 km, starts and ends near the centre |
| Trip towards the centre | 24 % | 13 min, 2.1 km, from the outskirts to about 1.1 km closer to the centre |
| Trip away from the centre | 14 % | 19 min, 3.1 km, from the centre to about 2.3 km further out |
| Long slow ride | 6 % | 92 min, 1.8 km from start to end |

The results suggest the following:
- Most trips seem to be short, direct trips from one point to another with weekday rush-hour peaks; long slow rides are a small part.
- The direction of the trip (towards or away from the centre) separates the trips more than the time of day.
- E-bikes appear more often in the longest direct trips (away from the centre), which could mean they are used to go
  further.
- Trips away from the centre more often start on a bike moved by the operator; this fits with bikes being brought
  back to the centre, but we cannot check it directly.
- The mix of profiles changes a lot between cities and countries, and so do the bike fleet (e-bikes) and the type
  of stations; these system differences probably explain part of the profile differences.

**Limitations**
- Only trips with a station at both ends are in the cleaned data (free-floating trips were removed), so the profiles
  describe station-based use only.
- There is no rider id: we group trips, not people, and the purpose of a trip (work, leisure…) is only a guess.
- Distance is a straight line between start and end, so loops and trips with stops look short and slow.
- The city centre is estimated as the median start point; for regional networks this may be far from the real centre.
- Some cities have few trips, and some countries only one or a few cities, so their profile mix is less reliable.
- Cargo bikes are only 0.06 % of the trips: too few to say anything about them.
- The data cover one year (Aug 2022 – Jul 2023) without weather or holidays, which could explain part of the patterns.
- The groups overlap (medium silhouette): they show tendencies, not strict types.